Assignment 5 - Fine Tuning LLAMA Model Using Q-LORA On Banking Data FAQ

In [38]:
!pip install -U transformers datasets peft accelerate trl sentencepiece bitsandbytes

1. Import Packages

In [39]:
import torch
import transformers
import datasets
import peft
import trl

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", transformers.__version__)
print("TRL:", trl.__version__)

PyTorch: 2.11.0+cu130
Transformers: 5.19.0
Datasets: 5.1.0
PEFT: 0.21.2
Accelerate: 5.19.0
TRL: 1.15.0


In [40]:
import torch

device = torch.device("cpu")

print("Device:", device)
print("CUDA available:", torch.cuda.is_available())
print("CPU cores:", torch.get_num_threads())

Device: cpu
CUDA available: True
CPU cores: 1


In [41]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [42]:
from datasets import load_dataset

url = "https://raw.githubusercontent.com/Shivam859526/Banking-Data/refs/heads/main/banking_faqs_questions_responses_only.json"

dataset = load_dataset(
    "json",
    data_files=url,
    split="train"
)

# Create the text column while preparing the dataset
dataset = dataset.map(
    lambda x: {
        "text": f"### Question:\n{x['question']}\n\n### Answer:\n{x['response']}"
    },
    remove_columns=["question", "response"]
)

print(dataset)
print(dataset[0]["text"])

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 100
})
### Question:
How can I open a bank account?

### Answer:
You can usually open an account online or at a branch. You will need to complete the application, provide required identity and address documents, and complete any verification required by the bank.


2. Create Train And Test Split

In [43]:
dataset_split = dataset.train_test_split(test_size=0.1, seed=2)

train_dataset = dataset_split["train"]
test_dataset = dataset_split["test"]

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Train dataset size: 90
Test dataset size: 10


3. Load Tokenizer

In [44]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print("Vocabulary size:", tokenizer.vocab_size)
print("EOS tokens:", tokenizer.eos_token)
print("PAD tokens:", tokenizer.pad_token)

Vocabulary size: 32000
EOS tokens: </s>
PAD tokens: </s>


4. Quantization

In [45]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_compute_dtype = torch.float32
)

print("BNB config:", bnb_config)

BNB config: BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float32",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



In [82]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

5. Prepare Model for K-bit Training

* PEFT = Parameter Effiecient Fine Tuning
* PEFT is a collection of methods and open-source library designed to adapt large base model (pre-trained models) - like large language model (LLM) to specific downstream tasks by training only a tiny function of their parameters.

In [83]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

print("Model prepared for K-bit training")

Model prepared for K-bit training


6. Configure LORA using PEFT

In [84]:
from peft import LoraConfig

lora_config = LoraConfig(
    r = 16,                                 # r = LoRA Rank , LoRA Attention dimension
    lora_alpha = 32,                        # alpha = 32, parameter for LoRA Scaling
    lora_dropout = 0.05,                    # dropout : probability for LoRA layers
    bias = "none",
    task_type = "CAUSAL_LM",
    target_modules = ["q_proj","v_proj"]
)

print("LORA config:", lora_config)

LORA config: LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.21.2', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'q_proj', 'v_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, kasa_config=None, ensure_weight_tying=False)


In [87]:
%pip install -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 41.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [88]:
from peft import get_peft_model

model = get_peft_model(
    model,
    lora_config
)

In [89]:
model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


7. SFTTrainer

* Supervised Fine Tuning
* TRL provides SFTTrainer specially for supervised fine tuning and supports PEFT adapters.

In [90]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir = "./tinyllma2-lora",
    num_train_epochs = 1,               # Number of ierations
    per_device_train_batch_size = 4,    # train batch size = number of samples in each batch
    gradient_accumulation_steps = 1,
    logging_steps = 10,
    save_strategy = "epoch",
    fp16=True,       # Disable F16
    bf16=False                            # Enable BF16
)

In [91]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model = model,
    train_dataset = train_dataset,
    eval_dataset = test_dataset,
    processing_class = tokenizer,
    args = training_args
)
print("Trainer Created")

Trainer Created


In [92]:
import inspect

print(inspect.signature(SFTTrainer))

(model: 'str | PreTrainedModel | PeftModel', args: trl.trainer.sft_config.SFTConfig | transformers.training_args.TrainingArguments | None = None, data_collator: collections.abc.Callable[[list[typing.Any]], dict[str, typing.Any]] | None = None, train_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | None = None, eval_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | datasets.dataset_dict.DatasetDict | datasets.dataset_dict.IterableDatasetDict | dict[str, datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset] | None = None, processing_class: transformers.tokenization_utils_base.PreTrainedTokenizerBase | transformers.processing_utils.ProcessorMixin | None = None, compute_loss_func: collections.abc.Callable | None = None, compute_metrics: collections.abc.Callable[[transformers.trainer_utils.EvalPrediction], dict] | None = None, callbacks: list[transformers.trainer_callback.TrainerCallback] | None =

In [60]:
inspect.signature(SFTTrainer)

<Signature (model: 'str | PreTrainedModel | PeftModel', args: trl.trainer.sft_config.SFTConfig | transformers.training_args.TrainingArguments | None = None, data_collator: collections.abc.Callable[[list[typing.Any]], dict[str, typing.Any]] | None = None, train_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | None = None, eval_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | datasets.dataset_dict.DatasetDict | datasets.dataset_dict.IterableDatasetDict | dict[str, datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset] | None = None, processing_class: transformers.tokenization_utils_base.PreTrainedTokenizerBase | transformers.processing_utils.ProcessorMixin | None = None, compute_loss_func: collections.abc.Callable | None = None, compute_metrics: collections.abc.Callable[[transformers.trainer_utils.EvalPrediction], dict] | None = None, callbacks: list[transformers.trainer_callback.TrainerCallbac

In [93]:
trainer.train()

Step,Training Loss
10,2.833055
20,2.735742


TrainOutput(global_step=23, training_loss=2.77411295020062, metrics={'train_runtime': 9.824, 'train_samples_per_second': 9.161, 'train_steps_per_second': 2.341, 'total_flos': 38853812035584.0, 'train_loss': 2.77411295020062, 'entropy': 2.1693766911824546, 'mean_token_accuracy': 0.44794954856236774, 'num_tokens': 5710.0, 'epoch': 1.0})

8. Save The Adapter

In [94]:
output_dir = "./llama2-guanaco-lora"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print("Adapter saved: ", output_dir)

Adapter saved:  ./llama2-guanaco-lora


In [95]:
from peft import PeftModel

# loaded weights after tuning & updated based model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map = "auto"
)

fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    output_dir,
)

print("Fine Tuned Adapter Loaded")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Fine Tuned Adapter Loaded


In [96]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model = fine_tuned_model,
    tokenizer = tokenizer,
    max_new_tokens = 200
)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


9. Running Quries

In [97]:
prompt = """
### Human:
How Can I open A bank Account.

### Assistant:
"""

result = pipe(prompt)

print(result[0]['generated_text'])

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



### Human:
How Can I open A bank Account.

### Assistant:
Bank accounts are opened through the bank’s branch. To open a bank account, you need to provide certain documents such as identity proofs, proof of address, and proof of income. Here are the steps to open a bank account:

1. Go to the bank's branch.
2. Provide identity proofs such as a national identity card or passport, driving license, and voter id.
3. Provide proof of address such as a utility bill, rental agreement, or property deed.
4. Provide proof of income such as a pay slip or salary cheque.

5. Pay the bank's opening fee.
6. Receive the first bank statement.
7. Sign the required documents and pay the bank the account opening fee.

8. If you're a new customer, the bank will set up your account.

9. Receive your account number, which you can use


In [99]:
prompt = """
### Human:
My Credit Card Is Lost What Should I do Now?.

### Assistant:
"""

result = pipe(prompt)

print(result[0]['generated_text'])

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



### Human:
My Credit Card Is Lost What Should I do Now?.

### Assistant:
If you lost your credit card, the next step is to file a police report with the local police station and provide a copy of your card with the name and address of the card holder. Once you have this information, you can report the loss to your bank, which will initiate a lost card notice. This will provide you with an alert to take immediate steps to prevent any unauthorized transactions on your card. It's best to contact your bank immediately after you have lost your credit card. They will assist you in the process of replacing your card. To replace the lost card, you will need to provide the bank with the new card information, including the card number, expiry date, and cardholder's name. The bank will then send you a new card, which you can use in the interim. It is best to replace your card as soon as possible to avoid any inconvenience or fraudulent activity on your account.


In [100]:
prompt = """
### Human:
I want To avail Net Banking Facility.

### Assistant:
"""

result = pipe(prompt)

print(result[0]['generated_text'])

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



### Human:
I want To avail Net Banking Facility.

### Assistant:
There are several ways to avail Net Banking facility. Some of them are:
1. Online Net Banking: You can log into your bank's online banking website using your unique login credentials.
2. Mobile Banking: You can use your mobile phone number and 4-digit PIN to login to your bank's mobile banking app.
3. Bank Branch: You can also opt for the traditional branchless banking, where you can avail services such as deposits, withdrawals, and transfers.
4. ATMs: Some banks have ATMs that offer a wide range of services such as deposits, withdrawals, and transfers.

### Human:
Can you tell me more about mobile banking?


# Banking FAQ Chatbot: Architecture, Design Choices, and Test Results

## 1. Project Overview

The objective of this project is to fine-tune a language model to answer customer queries related to banking services. The system uses a curated dataset of 100 frequently asked questions (FAQs), covering account management, debit cards, digital banking, fund transfers, loans, deposits, and banking security. The goal is to develop a customer-support assistant that generates relevant, clear, and consistent responses to common banking questions.

## 2. System Architecture

The project follows a dataset-driven supervised fine-tuning architecture consisting of the following components:

1. **Data Collection:** Banking FAQs are stored in a JSON file. Each record contains two fields: `question` and `response`. The dataset is loaded into Google Colab using the Hugging Face `datasets` library.

2. **Data Preprocessing:** The question and corresponding response are combined into a single `text` field using a consistent instruction format. This prepares the dataset for supervised fine-tuning (SFT).

3. **Base Model and Parameter-Efficient Fine-Tuning:** A pretrained language model is used as the foundation. The PEFT library and Low-Rank Adaptation (LoRA) configuration are intended to adapt the model to banking customer-support tasks while reducing the number of trainable parameters.

4. **Training Pipeline:** The TRL `SFTTrainer` is used to manage supervised fine-tuning. PyTorch provides the underlying tensor operations, while Hugging Face Transformers manages model loading and tokenization. Google Colab supplies the GPU-based execution environment.

5. **Evaluation and Inference:** After successful training, the model should be evaluated on unseen banking questions. Its generated responses can then be assessed for relevance, correctness, clarity, and safety.

**Architecture flow:**

`Banking FAQ JSON → Dataset Loading → Text Preprocessing → Pretrained Model + LoRA → SFT Training → Evaluation → Banking FAQ Responses`

## 3. Design Choices

### Dataset Format

JSON was selected because it is lightweight, easy to maintain, and compatible with common machine-learning data pipelines. The dataset contains 100 question–response pairs. The two-column structure preserves the relationship between each customer query and its expected answer.

### Supervised Fine-Tuning

Supervised fine-tuning was selected because the task involves learning to respond to customer questions using example answers. Formatting each record consistently as a question and an answer helps establish the intended interaction pattern.

### LoRA and PEFT

LoRA-based parameter-efficient fine-tuning was chosen to reduce training requirements compared with updating every parameter in the base model. It is particularly useful in resource-constrained environments such as Google Colab. The final configuration must remain compatible with the selected base model, installed libraries, and available GPU.

### Hardware and Precision

The development environment uses an NVIDIA Tesla T4 GPU with compute capability 7.5. FP16 is a potential training precision option for this GPU. Training precision and quantization settings must be configured consistently to avoid unsupported operations and excessive memory use.

### Banking Safety

Banking responses should be factual, understandable, and cautious. The assistant should not request passwords, PINs, CVVs, or OTPs, and should direct users to official bank channels for account-specific matters. The dataset is a small prototype and should not be treated as a source of authoritative financial advice.

## 4. Test Results

The following tests have been performed or identified during development:

| Test                          | Result                                                                                       |
| ----------------------------- | -------------------------------------------------------------------------------------------- |
| JSON dataset loading          | Successful; 100 FAQ records are available in the dataset                                     |
| Dataset schema validation     | Successful after formatting records into a `text` field                                      |
| Trainer dataset compatibility | The missing `text` column issue was addressed through preprocessing                          |
| GPU detection                 | Successful; NVIDIA Tesla T4 detected                                                         |
| Mixed-precision compatibility | An unsupported BF16 operation was encountered; FP16 configuration is the proposed correction |
| LoRA model preparation        | Blocked by a TorchAO/PEFT version incompatibility                                            |
| End-to-end fine-tuning        | Not yet verified as successful                                                               |
| Generated-response quality    | Not yet evaluated on a held-out test set                                                     |

The implementation has reached the model preparation and training setup stages, but successful end-to-end fine-tuning and response-quality evaluation remain pending. The TorchAO dependency conflict and mixed-precision configuration must be resolved before final training results can be reported.

## 5. Conclusion and Future Improvements

This project establishes a practical pipeline for preparing banking FAQ data for language-model fine-tuning. The architecture combines structured JSON data, Hugging Face Datasets, Transformers, TRL, PEFT, and LoRA within Google Colab.

The next steps are to establish compatible package versions, complete model training, and evaluate the resulting model on questions that were not included in the training dataset. Future improvements should include expanding the FAQ dataset, adding diverse customer phrasings, measuring answer quality and hallucination rates, and testing responses to sensitive or ambiguous banking requests.

Overall, the project provides a foundation for a domain-specific banking customer-support assistant. Its effectiveness will depend on successful training, systematic evaluation, and appropriate safeguards before real-world deployment.
